# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will prioritize pages that have meaningful search visibility and a clear refresh opportunity.

A page gets a higher score when it has:
- high search impressions,
- low CTR compared with pages in a similar average-position range,
- and has not been updated recently.

### Reason codes

- `ctr_fix` — the page has below-median CTR for its position bucket.
- `stale_and_visible` — the page is old and has meaningful search visibility.
- `ctr_fix_and_stale` — both signals are present.
- `visible_only` — the page has meaningful visibility but no stronger action signal.
- `low_priority` — the page does not meet the visibility threshold.

This is a transparent decision-support baseline. It does not predict Google's algorithm or guarantee that a refresh will improve performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# Load the starter dataset
paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next(p for p in paths if p.exists())
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# Signal 1: CTR compared with average-position bucket
# ---------------------------------------------------------

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

ctr_signal = (
    df[
        (df["impressions_90d"] > 0) &
        (df["avg_position"] > 0)
    ]
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("\nSignal 1 — CTR vs average-position bucket")
print(ctr_signal.round(3).to_string(index=False))

# ---------------------------------------------------------
# Signal 2: search visibility / volume
# ---------------------------------------------------------

df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 99, 499, 1999, np.inf],
    labels=["0-99", "100-499", "500-1999", "2000+"]
)

volume_signal = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("\nSignal 2 — search-volume buckets")
print(volume_signal.round(2).to_string(index=False))
# ---------------------------------------------------------
# One-word verdicts
# ---------------------------------------------------------

# For CTR, compare the main ranking buckets where enough data exists.
ctr_means = (
    ctr_signal
    .set_index("position_bucket")["mean_ctr"]
)

if (
    ctr_means["4-10"] > ctr_means["11-20"] >
    ctr_means["21+"]
):
    ctr_verdict = "CONFIRMED"
else:
    ctr_verdict = "MIXED"

# Search volume is useful because it separates pages by
# the amount of observable search demand.
if (
    volume_signal["n"].sum() == len(df)
    and volume_signal["median_impressions"].is_monotonic_increasing
):
    volume_verdict = "CONFIRMED"
else:
    volume_verdict = "MIXED"

print("\nSignal verdicts")
print("CTR vs position:", ctr_verdict)
print("Search volume:", volume_verdict)

Dataset shape: (30000, 44)

Signal 1 — CTR vs average-position bucket
position_bucket     n  median_ctr  mean_ctr
            1-3  1141        0.00     2.714
           4-10 11842        0.16     0.651
          11-20  7273        0.10     0.323
            21+  8539        0.00     0.211

Signal 2 — search-volume buckets
impression_bucket     n  median_impressions
             0-99  7994                12.0
          100-499  5280               251.5
         500-1999  6511              1009.0
            2000+ 10215              6473.0

Signal verdicts
CTR vs position: CONFIRMED
Search volume: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Scoring rule

I score pages using only observable search and content signals.

- +3 points when CTR is below the median CTR for its average-position bucket.
- +2 points when the page is at least 91 days since its last update and has at least 500 impressions.
- +1 point when the page has at least 500 impressions.

The score is used to rank pages for review. Each page also receives a reason code and an action label so the ranking is easy to explain.

I do not use trend fields or existing product flags in the score.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the ranked action queue

work = df.copy()

# Position buckets
work["position_bucket"] = pd.cut(
    work["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Calculate the median CTR within each position bucket.
# Only use pages with search visibility and a valid position.
position_medians = (
    work[
        (work["impressions_90d"] > 0) &
        (work["avg_position"] > 0)
    ]
    .groupby("position_bucket", observed=False)["ctr"]
    .median()
)

work["position_median_ctr"] = work["position_bucket"].map(position_medians)

# Signal 1: CTR opportunity
work["ctr_fix_flag"] = (
    (work["impressions_90d"] > 0) &
    (work["avg_position"] > 0) &
    work["position_median_ctr"].notna() &
    (work["ctr"] < work["position_median_ctr"])
).astype(int)

# Signal 2: stale + visible
work["stale_flag"] = (
    work["days_since_last_update"] >= 91
).astype(int)

work["visible_flag"] = (
    work["impressions_90d"] >= 500
).astype(int)

work["stale_and_visible_flag"] = (
    (work["stale_flag"] == 1) &
    (work["visible_flag"] == 1)
).astype(int)

# Transparent baseline score
work["action_score"] = (
    3 * work["ctr_fix_flag"] +
    2 * work["stale_and_visible_flag"] +
    1 * work["visible_flag"]
)

# Reason code
def reason_code(row):
    if row["ctr_fix_flag"] and row["stale_and_visible_flag"]:
        return "ctr_fix_and_stale"
    elif row["ctr_fix_flag"]:
        return "ctr_fix"
    elif row["stale_and_visible_flag"]:
        return "stale_and_visible"
    elif row["visible_flag"]:
        return "visible_only"
    else:
        return "low_priority"

work["reason_code"] = work.apply(reason_code, axis=1)

# Action label
def action_label(row):
    if row["ctr_fix_flag"]:
        return "CTR_FIX"
    elif row["stale_and_visible_flag"]:
        return "REFRESH"
    elif row["visible_flag"]:
        return "MONITOR"
    else:
        return "LOW_PRIORITY"

work["action"] = work.apply(action_label, axis=1)

# Rank highest score first.
# Impressions break ties so pages with more observable search demand
# are reviewed first.
work = work.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

# Current decline signal is used only for evaluation,
# never for creating the score.
work["declining_proxy"] = (
    work["trend_direction"] == "down"
).astype(int)

# Precision@K
def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

precision_50 = precision_at_k(
    work["declining_proxy"],
    work["action_score"],
    50
)

base_rate = work["declining_proxy"].mean()

print("Rows ranked:", len(work))
print("Base rate of current decline proxy:", round(base_rate, 3))
print("Baseline Precision@50:", round(precision_50, 3))

print("\nAction counts:")
print(work["action"].value_counts())

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Output required by the assignment
output_columns = [
    "rank",
    "content_id",
    "client_id",
    "action_score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

queue = work[output_columns].copy()

output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("\nCSV written to:", output_path)

print("\nTop 10 ranked pages:")
print(queue.head(10).to_string(index=False))

Rows ranked: 30000
Base rate of current decline proxy: 0.542
Baseline Precision@50: 0.48

Action counts:
action
CTR_FIX         9400
MONITOR         7951
LOW_PRIORITY    7533
REFRESH         5116
Name: count, dtype: int64

CSV written to: work\outputs\baseline_action_score.csv

Top 10 ranked pages:
 rank           content_id         client_id  action_score  action       reason_code  impressions_90d  ctr  avg_position  days_since_last_update
    1 content_5fe46e04994d client_4e07408562             6 CTR_FIX ctr_fix_and_stale           517715 0.14           4.2                     104
    2 content_36ff89c8214e client_19581e27de             6 CTR_FIX ctr_fix_and_stale           295097 0.05           7.3                     104
    3 content_c8e9d6ab9013 client_19581e27de             6 CTR_FIX ctr_fix_and_stale           208678 0.00           9.7                     104
    4 content_a7427266c305 client_19581e27de             6 CTR_FIX ctr_fix_and_stale           201111 0.11           5.7

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the twenty highest-ranked pages from the baseline queue. For each page, I record the action, reason code, confidence, and what could make the recommendation wrong.

These are review priorities based on observed signals, not guaranteed outcomes.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 manual review

top20 = work.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "ctr_fix_and_stale":
        return "Moderate — both CTR and staleness signals support review."
    elif row["reason_code"] == "ctr_fix":
        return "Moderate — CTR is below the position-bucket median."
    elif row["reason_code"] == "stale_and_visible":
        return "Moderate — the page is stale and has meaningful visibility."
    else:
        return "Low-moderate — the ranking is based mainly on visibility."

def wrong_reason(row):
    if row["reason_code"] == "ctr_fix_and_stale":
        return "The CTR gap may reflect SERP layout or query mix rather than content quality."
    elif row["reason_code"] == "ctr_fix":
        return "The low CTR may be caused by SERP features or query mix."
    elif row["reason_code"] == "stale_and_visible":
        return "The page may be intentionally evergreen or the update date may be incomplete."
    else:
        return "High visibility alone does not prove that the page needs a content change."

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "action_score",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

review["confidence"] = top20.apply(
    confidence_note,
    axis=1
).values

review["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
).values

print("Top-20 review")
print("=" * 120)

for _, row in review.iterrows():
    print(
        f"#{int(row['rank'])} | "
        f"{row['content_id']} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Score: {row['action_score']} | "
        f"Confidence: {row['confidence']} | "
        f"What could make it wrong: {row['what_would_make_it_wrong']}"
    )

print("\nStructured review table:")
display(review)

Top-20 review
#1 | content_5fe46e04994d | Action: CTR_FIX | Reason: ctr_fix_and_stale | Score: 6 | Confidence: Moderate — both CTR and staleness signals support review. | What could make it wrong: The CTR gap may reflect SERP layout or query mix rather than content quality.
#2 | content_36ff89c8214e | Action: CTR_FIX | Reason: ctr_fix_and_stale | Score: 6 | Confidence: Moderate — both CTR and staleness signals support review. | What could make it wrong: The CTR gap may reflect SERP layout or query mix rather than content quality.
#3 | content_c8e9d6ab9013 | Action: CTR_FIX | Reason: ctr_fix_and_stale | Score: 6 | Confidence: Moderate — both CTR and staleness signals support review. | What could make it wrong: The CTR gap may reflect SERP layout or query mix rather than content quality.
#4 | content_a7427266c305 | Action: CTR_FIX | Reason: ctr_fix_and_stale | Score: 6 | Confidence: Moderate — both CTR and staleness signals support review. | What could make it wrong: The CTR gap may refl

,rank,content_id,action,reason_code,action_score,impressions_90d,ctr,avg_position,days_since_last_update,confidence,what_would_make_it_wrong
0,1,content_5fe46e04994d,CTR_FIX,ctr_fix_and_stale,6,517715,0.14,4.2,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
1,2,content_36ff89c8214e,CTR_FIX,ctr_fix_and_stale,6,295097,0.05,7.3,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
2,3,content_c8e9d6ab9013,CTR_FIX,ctr_fix_and_stale,6,208678,0.00,9.7,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
3,4,content_a7427266c305,CTR_FIX,ctr_fix_and_stale,6,201111,0.11,5.7,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
4,5,content_91652435f57a,CTR_FIX,ctr_fix_and_stale,6,159590,0.06,7.8,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
5,6,content_f42eb861c6dd,CTR_FIX,ctr_fix_and_stale,6,152467,0.13,6.5,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
6,7,content_11fcfd65d94c,CTR_FIX,ctr_fix_and_stale,6,149083,0.15,6.2,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
7,8,content_97a86caf3a3d,CTR_FIX,ctr_fix_and_stale,6,147670,0.07,6.4,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
8,9,content_c1fe78bc4e37,CTR_FIX,ctr_fix_and_stale,6,134055,0.03,7.5,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...
9,10,content_4c76e9b13aea,CTR_FIX,ctr_fix_and_stale,6,127952,0.07,7.4,104,Moderate — both CTR and staleness signals supp...,The CTR gap may reflect SERP layout or query m...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some lower-ranked pages can be weak picks because the baseline is intentionally simple. High impressions alone do not prove that a page needs a refresh, and a low CTR can be affected by query mix or SERP features.

The top-ranked pages are also tied on the same score and staleness value, so impressions are used to break the tie. This means the exact order within a tied score should not be treated as a strong distinction.

### Leakage check

I checked that the action score does not use product-generated flags, the decline label, or future-window information.

`trend_direction` is used only as the evaluation proxy after ranking. It is not used to calculate the score.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Weak picks
# ---------------------------------------------------------

weak_picks = work[
    (work["action_score"] <= 1) &
    (work["impressions_90d"] >= 500)
].head(5)

print("Examples of weaker picks:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "action",
            "reason_code",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].to_string(index=False)
)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

# These fields must not be used to construct the score.
forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining",
    "needs_ctr_fix",
    "is_quick_win",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_initial_refresh_candidate",
    "health_score"
]

# These are the only signals used to construct the score.
score_components = [
    "ctr_fix_flag",
    "stale_and_visible_flag",
    "visible_flag"
]

# Confirm that no forbidden field is part of the scoring components.
leaked_score_fields = [
    field for field in forbidden_fields
    if field in score_components
]

print("\nForbidden fields used in score:", leaked_score_fields)

assert leaked_score_fields == []

# Confirm that the decline proxy is only used for evaluation.
print(
    "Decline proxy used for evaluation only:",
    "declining_proxy" in work.columns
)

print("\nLeakage check passed.")
print("No product flags, trend fields, or future-window fields are used in the score.")

Examples of weaker picks:
 rank           content_id  action_score  action  reason_code  impressions_90d  ctr  avg_position  days_since_last_update
14517 content_aaef01a50def             1 MONITOR visible_only           517109 0.25           5.4                      22
14518 content_8c19996aa890             1 MONITOR visible_only           509252 0.15           2.5                      20
14519 content_2cb567c3c89b             1 MONITOR visible_only           497727 0.10          22.2                      48
14520 content_4c36c775b818             1 MONITOR visible_only           463103 0.41           2.3                      20
14521 content_1a9e894be2e2             1 MONITOR visible_only           416180 0.23           4.0                      22

Forbidden fields used in score: []
Decline proxy used for evaluation only: True

Leakage check passed.
No product flags, trend fields, or future-window fields are used in the score.


### Named limitation

The baseline produces many tied scores. In the top-20 review, all pages have the same score of 6 and the same staleness value of 104 days, so impressions determine their order. Therefore, small differences in rank within a tied score should not be interpreted as meaningful differences in review priority.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.